In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    roc_auc_score, roc_curve
)
from sklearn.model_selection import GridSearchCV, cross_val_score
import pickle
import joblib

In [ ]:
# load the dataset
X_train = joblib.load("../Preprocessing/X_train.pkl")
X_test = joblib.load("../Preprocessing/X_test.pkl")
y_train = joblib.load("../Preprocessing/y_train.pkl")
y_test = joblib.load("../Preprocessing/y_test.pkl")

print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")

In [ ]:
# Baseline model

baseline_xgb = XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='binary:logistic',
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1,
    scale_pos_weight=1
)

baseline_xgb.fit(X_train, y_train)

y_pred_baseline = baseline_xgb.predict(X_test)
y_prob_baseline = baseline_xgb.predict_proba(X_test)[:, 1]

baseline_acc = accuracy_score(y_test, y_pred_baseline)
baseline_auc = roc_auc_score(y_test, y_prob_baseline)

print(f"Baseline Test Accuracy: {baseline_acc:.4f}")
print(f"Baseline Test AUC: {baseline_auc:.4f}")

In [ ]:
# Hyperparameter Tuning

# Focused parameter grid for XGBoost
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.2],
    'subsample': [0.8, 0.9],
    'colsample_bytree': [0.8, 0.9],
    'min_child_weight': [1, 3],
    'gamma': [0, 0.1]
}

grid_search = GridSearchCV(
    estimator=XGBClassifier(
        objective='binary:logistic',
        eval_metric='logloss',
        random_state=42,
        n_jobs=-1
    ),
    param_grid=param_grid,
    cv=3,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

print(f"\nBest parameters:")
for param, value in grid_search.best_params_.items():
    print(f"  {param}: {value}")
print(f"\nBest CV AUC: {grid_search.best_score_:.4f}")

In [ ]:
# Optimized Model Evaluation

optimized_model = grid_search.best_estimator_

# Predictions
y_train_pred = optimized_model.predict(X_train)
y_test_pred = optimized_model.predict(X_test)
y_train_prob = optimized_model.predict_proba(X_train)[:, 1]
y_test_prob = optimized_model.predict_proba(X_test)[:, 1]

# Metrics
train_acc = accuracy_score(y_train, y_train_pred)
test_acc = accuracy_score(y_test, y_test_pred)
train_auc = roc_auc_score(y_train, y_train_prob)
test_auc = roc_auc_score(y_test, y_test_prob)

print(f"\nTraining Accuracy: {train_acc:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")
print(f"Training AUC: {train_auc:.4f}")
print(f"Test AUC: {test_auc:.4f}")

print(f"\nImprovement:")
print(f"  Accuracy: {baseline_acc:.4f} → {test_acc:.4f} ({(test_acc - baseline_acc)*100:+.2f}%)")
print(f"  AUC: {baseline_auc:.4f} → {test_auc:.4f} ({(test_auc - baseline_auc)*100:+.2f}%)")

print("\nClassification Report")
print(classification_report(y_test, y_test_pred, target_names=['No Attrition', 'Attrition']))

# Cross-validation scores
cv_scores = cross_val_score(optimized_model, X_train, y_train, cv=5, scoring='roc_auc')
print(f"\nCross-Validation Scores")
print(f"CV AUC Scores: {[f'{score:.4f}' for score in cv_scores]}")
print(f"Mean CV AUC: {cv_scores.mean():.4f} (+/- {cv_scores.std() * 2:.4f})")


In [ ]:
# Visualizations
fig = plt.figure(figsize=(16, 10))

# Confusion Matrix - Baseline
plt.subplot(2, 3, 1)
cm_baseline = confusion_matrix(y_test, y_pred_baseline)
sns.heatmap(cm_baseline, annot=True, fmt='d', cmap='Reds',
            xticklabels=['No Attrition', 'Attrition'],
            yticklabels=['No Attrition', 'Attrition'])
plt.title(f'Baseline Model\nAcc: {baseline_acc:.4f}, AUC: {baseline_auc:.4f}',
          fontweight='bold', fontsize=11)
plt.ylabel('Actual')
plt.xlabel('Predicted')

In [ ]:
fig = plt.figure(figsize=(16, 10))

# Confusion Matrix - Optimized
plt.subplot(2, 3, 1)
cm_optimized = confusion_matrix(y_test, y_test_pred)
sns.heatmap(cm_optimized, annot=True, fmt='d', cmap='Greens',
            xticklabels=['No Attrition', 'Attrition'],
            yticklabels=['No Attrition', 'Attrition'])
plt.title(f'Optimized Model\nAcc: {test_acc:.4f}, AUC: {test_auc:.4f}',
          fontweight='bold', fontsize=11)
plt.ylabel('Actual')
plt.xlabel('Predicted')

In [ ]:
fig = plt.figure(figsize=(16, 10))

# ROC Curve Comparison
plt.subplot(2, 3, 3)
fpr_baseline, tpr_baseline, _ = roc_curve(y_test, y_prob_baseline)
fpr_optimized, tpr_optimized, _ = roc_curve(y_test, y_test_prob)

plt.plot(fpr_baseline, tpr_baseline, label=f'Baseline (AUC={baseline_auc:.4f})',
         linewidth=2.5, alpha=0.8)
plt.plot(fpr_optimized, tpr_optimized, label=f'Optimized (AUC={test_auc:.4f})',
         linewidth=2.5, alpha=0.8)
plt.plot([0, 1], [0, 1], 'k--', label='Random', alpha=0.5)
plt.xlabel('False Positive Rate', fontweight='bold')
plt.ylabel('True Positive Rate', fontweight='bold')
plt.title('ROC Curve Comparison', fontweight='bold', fontsize=12)
plt.legend(loc='lower right')
plt.grid(alpha=0.3)

In [ ]:
fig = plt.figure(figsize=(16, 10))

# Feature Importance
plt.subplot(2, 3, 4)
feature_importance = pd.DataFrame({
    'feature': X_train.columns,
    'importance': optimized_model.feature_importances_
}).sort_values('importance', ascending=False)

top_features = feature_importance.head(15)
colors = plt.cm.viridis(np.linspace(0.3, 0.9, len(top_features)))
plt.barh(range(len(top_features)), top_features['importance'], color=colors, edgecolor='black')
plt.yticks(range(len(top_features)), top_features['feature'])
plt.xlabel('Importance', fontweight='bold')
plt.title('Top 15 Feature Importances', fontweight='bold', fontsize=12)
plt.gca().invert_yaxis()

In [ ]:
fig = plt.figure(figsize=(16, 10))

# Performance Comparison
plt.subplot(2, 3, 5)
metrics = ['Accuracy', 'AUC']
baseline_scores = [baseline_acc, baseline_auc]
optimized_scores = [test_acc, test_auc]

x = np.arange(len(metrics))
width = 0.35

bars1 = plt.bar(x - width/2, baseline_scores, width, label='Baseline',
                color='lightcoral', alpha=0.8, edgecolor='black')
bars2 = plt.bar(x + width/2, optimized_scores, width, label='Optimized',
                color='lightgreen', alpha=0.8, edgecolor='black')

plt.ylabel('Score', fontweight='bold')
plt.title('Model Performance Comparison', fontweight='bold', fontsize=12)
plt.xticks(x, metrics)
plt.legend()
plt.ylim([0.90, 1.0])

for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.4f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

In [ ]:
fig = plt.figure(figsize=(16, 10))

# Cross-Validation Scores
plt.subplot(2, 3, 6)
plt.plot(range(1, 6), cv_scores, marker='o', linestyle='-', linewidth=2.5,
         markersize=10, color='steelblue')
plt.axhline(y=cv_scores.mean(), color='r', linestyle='--', linewidth=2,
            label=f'Mean: {cv_scores.mean():.4f}')
plt.fill_between(range(1, 6),
                 cv_scores.mean() - cv_scores.std(),
                 cv_scores.mean() + cv_scores.std(),
                 alpha=0.2, color='steelblue')
plt.xlabel('Fold', fontweight='bold')
plt.ylabel('AUC Score', fontweight='bold')
plt.title('Cross-Validation Scores', fontweight='bold', fontsize=12)
plt.legend()
plt.grid(alpha=0.3)
plt.ylim([cv_scores.min() - 0.01, cv_scores.max() + 0.01])

plt.tight_layout()
plt.show()

# Additional Visualization - Top Hyperparameter Configurations
fig2 = plt.figure(figsize=(14, 5))

# Get top 20 configurations
results_df = pd.DataFrame(grid_search.cv_results_)
top_configs = results_df.nsmallest(20, 'rank_test_score')

plt.subplot(1, 2, 1)
plt.scatter(range(len(top_configs)), top_configs['mean_test_score'],
           s=200, c=top_configs['mean_test_score'], cmap='RdYlGn',
           edgecolor='black', alpha=0.7, linewidth=1.5)
plt.colorbar(label='Mean CV AUC')
plt.axhline(y=grid_search.best_score_, color='red', linestyle='--',
            linewidth=2, label=f'Best: {grid_search.best_score_:.4f}')
plt.xlabel('Configuration Rank', fontweight='bold')
plt.ylabel('Mean CV AUC Score', fontweight='bold')
plt.title('Top 20 Configurations Performance', fontweight='bold', fontsize=12)
plt.legend()
plt.grid(alpha=0.3)

plt.subplot(1, 2, 2)
plt.axis('off')

In [ ]:
# Summary table
summary_data = [
    ['Metric', 'Baseline', 'Optimized', 'Change'],
    ['Test Accuracy', f'{baseline_acc:.4f}', f'{test_acc:.4f}',
     f'{(test_acc - baseline_acc)*100:+.2f}%'],
    ['Test AUC', f'{baseline_auc:.4f}', f'{test_auc:.4f}',
     f'{(test_auc - baseline_auc)*100:+.2f}%'],
    ['Train Accuracy', '1.0000', f'{train_acc:.4f}',
     f'{(train_acc - 1.0)*100:+.2f}%'],
    ['CV AUC (mean)', 'N/A', f'{cv_scores.mean():.4f}', 'N/A']
]

table = plt.table(cellText=summary_data, cellLoc='center', loc='center',
                 colWidths=[0.28, 0.24, 0.24, 0.24])
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1, 2.5)

# Header formatting
for i in range(4):
    table[(0, i)].set_facecolor('#2196F3')
    table[(0, i)].set_text_props(weight='bold', color='white')

# First column
for i in range(1, 5):
    table[(i, 0)].set_facecolor('#E3F2FD')

plt.title('Performance Summary', fontweight='bold', pad=20, fontsize=14)

plt.tight_layout()
plt.show()


print("BEST HYPERPARAMETERS")
for param, value in sorted(grid_search.best_params_.items()):
    print(f"  {param:20s}: {value}")

In [ ]:
# Save models and results

with open("xgboost_model_optimized.pkl", "wb") as f:
    pickle.dump(optimized_model, f)
print("Optimized XGBoost model saved as 'xgboost_model_optimized.pkl'")